# 06 - Gmail via OAuth (send-only scope)

Same `get_weather` + `send_email` combo as `05_weather_email.ipynb`, but `send_email` now goes through the real Gmail API with proper OAuth instead of an SMTP App Password. Heavier setup, but this is what a production integration would actually use, and it only grants the `gmail.send` scope - it cannot read your mail.

**Prerequisites (you said you already have these):**
1. A Google Cloud project with the **Gmail API** enabled.
2. An OAuth 2.0 Client ID of type **Desktop app**, with its JSON downloaded.
3. That file uploaded into this notebook's folder, renamed to `client_secret.json` (never commit it - see `.gitignore`).

**Before running the OAuth cell below:** you need `gmail_token.json` already sitting in this folder. Get it by running **[`../get_token.py`](../get_token.py)** on your own computer (not this Codespace) - see the README's OAuth setup section for why, and the exact steps. The in-Codespace browser flow this cell attempts as a fallback frequently fails with `ERR_CONNECTION_REFUSED` unless you're using VS Code Desktop attached to the Codespace.

## 1. Install dependencies

In [1]:
%pip install -q anthropic requests google-api-python-client google-auth-httplib2 google-auth-oauthlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Set your Anthropic API key

Uses `getpass` so the key isn't saved into the notebook file.

In [5]:
import os
from getpass import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your ANTHROPIC_API_KEY: ")

# Only needed if you hit: "anthropic-workspace-id is required when
# authenticating with an identity-linked API key" - leave blank to skip.
if not os.environ.get("ANTHROPIC_WORKSPACE_ID"):
    _workspace_id = input("ANTHROPIC_WORKSPACE_ID (leave blank if not needed): ").strip()
    if _workspace_id:
        os.environ["ANTHROPIC_WORKSPACE_ID"] = _workspace_id

## 3. Gmail OAuth

If `gmail_token.json` already exists in this folder (from running `get_token.py` on your own computer), this cell just loads and uses it - no browser needed.

Otherwise it falls back to opening an authorization URL for you to visit and starting a local server to catch the redirect. **This fallback usually doesn't work from a browser-based Codespace** - the redirect target is hardcoded to literal `localhost`, which only resolves back to this container when VS Code Desktop is tunneling it for you. If you hit `ERR_CONNECTION_REFUSED`, don't keep retrying this cell - go run `get_token.py` locally instead (see the notebook intro above).

In [4]:
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build as build_google_service

GMAIL_SCOPES = ["https://www.googleapis.com/auth/gmail.send"]
CLIENT_SECRET_FILE = "client_secret.json"
TOKEN_FILE = "gmail_token.json"


def get_gmail_service():
    """OAuth flow for the Gmail API - send-only scope, least privilege.

    First run: opens a URL for you to authorize in your own browser, then
    starts a local server (on an OS-assigned free port, so it never
    collides with something already running) to catch the redirect. In a
    Codespace, watch for an "Open in Browser" port-forwarding notification
    if the redirect doesn't complete on its own.

    Later runs: reuses gmail_token.json (refreshing it if expired), so you
    only go through the browser step once.
    """
    creds = None
    if os.path.exists(TOKEN_FILE):
        creds = Credentials.from_authorized_user_file(TOKEN_FILE, GMAIL_SCOPES)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                CLIENT_SECRET_FILE, GMAIL_SCOPES
            )
            creds = flow.run_local_server(port=0, open_browser=False)
        with open(TOKEN_FILE, "w") as f:
            f.write(creds.to_json())

    return build_google_service("gmail", "v1", credentials=creds)


gmail_service = get_gmail_service()
print("Gmail OAuth ready.")


Gmail OAuth ready.


## 4. Tools, the agent loop, and the `Agent` class

Same cost cap and turn-collapsing core as the other notebooks. `send_email` now calls the Gmail API via `gmail_service` from the cell above instead of smtplib.

In [6]:
import datetime
import json
import os

import anthropic

MODEL = "claude-haiku-4-5"
MAX_TOKENS = int(os.environ.get("AGENT_MAX_TOKENS", "1024"))

# claude-haiku-4-5 pricing, $/1M tokens - update if you switch models.
INPUT_COST_PER_MTOK = 1.00
OUTPUT_COST_PER_MTOK = 5.00

# Hard spending cap for this notebook kernel session.
MAX_COST_USD = float(os.environ.get("AGENT_MAX_COST_USD", "0.20"))


class BudgetExceededError(RuntimeError):
    pass
import base64
from email.message import EmailMessage

import requests

SYSTEM_PROMPT = (
    "You are a helpful assistant with get_weather and send_email tools. "
    "Use get_weather for current conditions. Only call send_email when the "
    "user explicitly asks you to send or email something - never send "
    "email on your own initiative. Otherwise reply directly."
)

TOOLS = [
    {
        "name": "get_weather",
        "description": "Get current weather conditions for a location by city name.",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City name, optionally with country, e.g. 'Paris, France'.",
                },
            },
            "required": ["location"],
        },
    },
    {
        "name": "send_email",
        "description": "Send a plain-text email via the Gmail API to a recipient.",
        "input_schema": {
            "type": "object",
            "properties": {
                "to": {"type": "string", "description": "Recipient email address."},
                "subject": {"type": "string", "description": "Email subject line."},
                "body": {"type": "string", "description": "Plain-text email body."},
            },
            "required": ["to", "subject", "body"],
        },
    },
]

# WMO weather interpretation codes (used by Open-Meteo's weather_code field).
WMO_CODES = {
    0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
    45: "Fog", 48: "Depositing rime fog",
    51: "Light drizzle", 53: "Moderate drizzle", 55: "Dense drizzle",
    56: "Light freezing drizzle", 57: "Dense freezing drizzle",
    61: "Slight rain", 63: "Moderate rain", 65: "Heavy rain",
    66: "Light freezing rain", 67: "Heavy freezing rain",
    71: "Slight snow fall", 73: "Moderate snow fall", 75: "Heavy snow fall",
    77: "Snow grains",
    80: "Slight rain showers", 81: "Moderate rain showers", 82: "Violent rain showers",
    85: "Slight snow showers", 86: "Heavy snow showers",
    95: "Thunderstorm", 96: "Thunderstorm with slight hail", 99: "Thunderstorm with heavy hail",
}


def _get_with_retry(url: str, params: dict, attempts: int = 2):
    last_exc = None
    for attempt in range(attempts):
        try:
            resp = requests.get(url, params=params, timeout=20)
            resp.raise_for_status()
            return resp
        except requests.RequestException as exc:
            last_exc = exc
    raise last_exc


def get_weather(location: str) -> str:
    """Free, no-API-key weather lookup via Open-Meteo."""
    try:
        geo_resp = _get_with_retry(
            "https://geocoding-api.open-meteo.com/v1/search",
            {"name": location, "count": 1},
        )
        geo_data = geo_resp.json()
    except requests.RequestException as exc:
        return f"Error: location lookup failed ({exc})"

    results = geo_data.get("results")
    if not results:
        return f"Error: could not find a location matching '{location}'."
    place = results[0]

    try:
        weather_resp = _get_with_retry(
            "https://api.open-meteo.com/v1/forecast",
            {
                "latitude": place["latitude"],
                "longitude": place["longitude"],
                "current": "temperature_2m,wind_speed_10m,weather_code",
            },
        )
        weather_data = weather_resp.json()
    except requests.RequestException as exc:
        return f"Error: weather lookup failed ({exc})"

    current = weather_data.get("current", {})
    units = weather_data.get("current_units", {})
    condition = WMO_CODES.get(current.get("weather_code"), "Unknown conditions")
    place_label = place["name"] + ((", " + place["country"]) if place.get("country") else "")

    return (
        "Weather in " + place_label + ": " + condition + ", "
        + str(current.get("temperature_2m")) + units.get("temperature_2m", "\u00b0C")
        + ", wind " + str(current.get("wind_speed_10m")) + " " + units.get("wind_speed_10m", "km/h")
    )


def send_email(to: str, subject: str, body: str) -> str:
    """Send via the Gmail API using the OAuth-authorized gmail_service
    (see the OAuth setup cell above)."""
    message = EmailMessage()
    message.set_content(body)
    message["To"] = to
    message["Subject"] = subject

    encoded = base64.urlsafe_b64encode(message.as_bytes()).decode()
    try:
        gmail_service.users().messages().send(
            userId="me", body={"raw": encoded}
        ).execute()
        return f"Email sent to {to}."
    except Exception as exc:
        return f"Error: failed to send email ({exc})"


def execute_tool(name: str, tool_input: dict) -> str:
    if name == "get_weather":
        return get_weather(tool_input["location"])
    if name == "send_email":
        return send_email(tool_input["to"], tool_input["subject"], tool_input["body"])
    return f"Error: unknown tool '{name}'"


MAX_PAUSE_RESUMES = 10


def build_client() -> anthropic.Anthropic:
    """Some API keys (personal keys not scoped to one workspace) require an
    anthropic-workspace-id header on every request - see
    https://platform.claude.com/docs/en/manage-claude/authentication#select-a-workspace.
    Set ANTHROPIC_WORKSPACE_ID if you hit: 'anthropic-workspace-id is
    required when authenticating with an identity-linked API key'."""
    workspace_id = os.environ.get("ANTHROPIC_WORKSPACE_ID")
    if workspace_id:
        return anthropic.Anthropic(
            default_headers={"anthropic-workspace-id": workspace_id}
        )
    return anthropic.Anthropic()


class Agent:
    """A minimal conversational agent that can call tools in a loop."""

    def __init__(self, client: anthropic.Anthropic | None = None):
        self.client = client or build_client()
        self.messages: list[dict] = []
        self.total_cost_usd = 0.0

    def send(self, user_input: str) -> str:
        turn_start = len(self.messages)
        self.messages.append({"role": "user", "content": user_input})

        resumes = 0
        while True:
            if self.total_cost_usd >= MAX_COST_USD:
                raise BudgetExceededError(
                    f"Session cost ${self.total_cost_usd:.4f} has reached the "
                    f"${MAX_COST_USD:.4f} cap (AGENT_MAX_COST_USD). Raise the "
                    "cap or start a new session to continue."
                )

            response = self.client.messages.create(
                model=MODEL,
                max_tokens=MAX_TOKENS,
                system=SYSTEM_PROMPT,
                tools=TOOLS,
                messages=self.messages,
            )
            self.total_cost_usd += (
                response.usage.input_tokens * INPUT_COST_PER_MTOK
                + response.usage.output_tokens * OUTPUT_COST_PER_MTOK
            ) / 1_000_000
            self.messages.append({"role": "assistant", "content": response.content})

            if response.stop_reason == "pause_turn":
                resumes += 1
                if resumes > MAX_PAUSE_RESUMES:
                    break
                continue

            if response.stop_reason != "tool_use":
                break

            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = execute_tool(block.name, block.input)
                    tool_results.append(
                        {
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": result,
                        }
                    )
            self.messages.append({"role": "user", "content": tool_results})

        reply = "".join(
            block.text for block in response.content if block.type == "text"
        )
        self.messages[turn_start:] = [
            {"role": "user", "content": user_input},
            {"role": "assistant", "content": reply},
        ]
        return reply


## 5. Create the agent

In [7]:
agent = Agent()
print(f"Agent ready (cap ${MAX_COST_USD:.4f} for this kernel session)")

Agent ready (cap $0.2000 for this kernel session)


## 6. Try it

Replace `you@example.com` with your own address before running - this really sends an email.

In [8]:
reply = agent.send(
    "What's the weather in Tokyo, and send it to suresh6312@gmail.com "
    "with the subject 'Tokyo Weather'."
)
print(reply)
print(f"(session cost so far: ~${agent.total_cost_usd:.4f})")

Done! I've retrieved the weather for Tokyo and sent it to suresh6312@gmail.com. The current conditions in Tokyo are:
- **Light drizzle**
- **Temperature**: 23.0°C
- **Wind**: 2.4 km/h
(session cost so far: ~$0.0039)


## 7. Optional: interactive chat loop

Type `exit` to stop.

In [ ]:
while True:
    user_input = input("You: ")
    if user_input.strip().lower() in {"exit", "quit"}:
        break
    try:
        reply = agent.send(user_input)
    except BudgetExceededError as exc:
        print(f"Agent: [stopped] {exc}")
        break
    print(f"Agent: {reply}  (session cost so far: ~${agent.total_cost_usd:.4f})")
